# Round 3 notebook 05 — forensic bot edges

        This is the second-pass notebook: the goal is not another broad summary, but a stricter search for
        hidden exploitable behavior.

        Focus areas:

        - HYDROGEL hidden bot roles and fillable passive edge under position limits.
        - Whether HYDROGEL quote sizes, L3 levels, and slow state encode future moves.
        - Whether the OTM voucher seller is one bot, many bots, or a coordinated strip process.
        - Which attractive-looking patterns fail a holdout check and should not drive the next strategy.

        No `trader.py` changes are made here.

In [1]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from pathlib import Path
import math
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

DATA = Path("data/ROUND3")
DAYS = [0, 1, 2]

def load_round3():
    prices = []
    trades = []
    for day in DAYS:
        p = pd.read_csv(DATA / f"prices_round_3_day_{day}.csv", sep=";")
        t = pd.read_csv(DATA / f"trades_round_3_day_{day}.csv", sep=";")
        p["day"] = day
        t["day"] = day
        prices.append(p)
        trades.append(t)
    prices = pd.concat(prices, ignore_index=True)
    trades = pd.concat(trades, ignore_index=True)
    for col in prices.columns:
        if col.startswith(("bid_", "ask_")) or col == "mid_price":
            prices[col] = pd.to_numeric(prices[col], errors="coerce")
    trades["price"] = pd.to_numeric(trades["price"], errors="coerce")
    trades["quantity"] = pd.to_numeric(trades["quantity"], errors="coerce")
    return prices, trades

def add_book_features(prices):
    p = prices.copy()
    p["best_bid"] = p["bid_price_1"]
    p["best_ask"] = p["ask_price_1"]
    p["spread"] = p["best_ask"] - p["best_bid"]
    bid_cols = ["bid_price_1", "bid_price_2", "bid_price_3"]
    ask_cols = ["ask_price_1", "ask_price_2", "ask_price_3"]
    p["wall_bid"] = p[bid_cols].min(axis=1)
    p["wall_ask"] = p[ask_cols].max(axis=1)
    p["wall_mid"] = (p["wall_bid"] + p["wall_ask"]) / 2
    p["wall_spread"] = p["wall_ask"] - p["wall_bid"]
    p["n_bid_levels"] = p[bid_cols].notna().sum(axis=1)
    p["n_ask_levels"] = p[ask_cols].notna().sum(axis=1)
    p["mid_minus_wall"] = p["mid_price"] - p["wall_mid"]
    return p

def add_future_mid(prices, horizons=(1, 5, 10, 25, 50, 100, 200, 500, 1000, 2000, 5000)):
    p = prices.sort_values(["product", "day", "timestamp"]).copy()
    for h in horizons:
        p[f"fut_mid_{h}"] = p.groupby(["product", "day"])["mid_price"].shift(-h)
        p[f"ret_{h}"] = p[f"fut_mid_{h}"] - p["mid_price"]
    return p

def attach_trades(prices, trades):
    t = trades.merge(
        prices,
        left_on=["day", "timestamp", "symbol"],
        right_on=["day", "timestamp", "product"],
        how="left",
    )
    t["side"] = np.where(
        t["price"] == t["ask_price_1"],
        "buy_agg",
        np.where(t["price"] == t["bid_price_1"], "sell_agg", "other"),
    )
    return t

def maker_markout_table(trades_with_book, horizons=(1, 10, 50, 100, 200, 500, 1000, 2000, 5000)):
    t = trades_with_book.copy()
    for h in horizons:
        # PnL for a passive maker who sold to an aggressive buyer, or bought from an aggressive seller.
        t[f"mk_{h}"] = np.where(
            t["side"] == "buy_agg",
            t["price"] - t[f"fut_mid_{h}"],
            np.where(t["side"] == "sell_agg", t[f"fut_mid_{h}"] - t["price"], np.nan),
        )
    agg = {
        "n": ("price", "size"),
        "avg_qty": ("quantity", "mean"),
        "avg_price": ("price", "mean"),
    }
    for h in horizons:
        agg[f"mk_{h}"] = (f"mk_{h}", "mean")
    return t.groupby(["symbol", "side"]).agg(**agg).round(3).reset_index()

def slow_ema_payoff(prices, span=1000, horizons=(10, 25, 50, 100, 200, 500, 1000, 2000)):
    rows = []
    p = prices.sort_values(["product", "day", "timestamp"]).copy()
    alpha = 2 / (span + 1)
    p["ema"] = p.groupby(["product", "day"])["mid_price"].transform(
        lambda s: s.ewm(alpha=alpha, adjust=False).mean()
    )
    p["dev"] = p["mid_price"] - p["ema"]
    for product, g in p.groupby("product"):
        if g["mid_price"].std() == 0:
            rows.append({"product": product, "dev_q10": 0, "dev_q90": 0, "best_H": None, "edge_ticks": 0})
            continue
        lo, hi = g["dev"].quantile(0.10), g["dev"].quantile(0.90)
        low = g[g["dev"] <= lo]
        high = g[g["dev"] >= hi]
        best = None
        for h in horizons:
            low_ret = low.groupby("day")["mid_price"].shift(-h) if False else low[f"ret_{h}"]
            high_ret = high[f"ret_{h}"]
            edge = (low_ret.mean() - high_ret.mean()) / 2
            rec = (h, edge, low_ret.mean(), high_ret.mean())
            if best is None or edge > best[1]:
                best = rec
        rows.append({
            "product": product,
            "dev_q10": lo,
            "dev_q90": hi,
            "best_H": best[0],
            "edge_ticks": best[1],
            "low_future_move": best[2],
            "high_future_move": best[3],
        })
    return pd.DataFrame(rows).sort_values("edge_ticks", ascending=False).round(3)

In [2]:
prices_raw, trades_raw = load_round3()
prices = add_future_mid(add_book_features(prices_raw), horizons=(1, 10, 50, 100, 200, 500, 1000, 2000, 5000))

def add_ema_devs(df, spans=(500, 1000, 2000, 5000)):
    out = df.sort_values(["product", "day", "timestamp"]).copy()
    for span in spans:
        alpha = 2 / (span + 1)
        out[f"ema{span}"] = out.groupby(["product", "day"])["mid_price"].transform(
            lambda s: s.ewm(alpha=alpha, adjust=False).mean()
        )
        out[f"dev{span}"] = out["mid_price"] - out[f"ema{span}"]
    return out

prices = add_ema_devs(prices)
trades = attach_trades(prices, trades_raw)
display(Markdown("Loaded R3 prices/trades with future marks and slow EMA deviations."))

Loaded R3 prices/trades with future marks and slow EMA deviations.

## 1. HYDROGEL: passive taker-capture replay

        This replay is deliberately conservative about *where* the edge comes from:

        - Only fills when the historical trade file shows an L1 taker at that timestamp.
        - If a seller hits the bid, assume our improved bid at `bid+1` can buy up to the taker quantity.
        - If a buyer lifts the ask, assume our improved ask at `ask-1` can sell up to the taker quantity.
        - Enforce the 200-lot HYDROGEL limit.
        - Reset per historical day and liquidate residual inventory at day-end mid.

        This is not a prosperity4btx score. It is a website-like passive-flow model for deciding which side
        we should want to be on when the hidden taker bots arrive.

In [3]:
h = prices[prices["product"] == "HYDROGEL_PACK"].copy().sort_values(["day", "timestamp"])
h["l3flag"] = np.select(
    [h["ask_price_3"].notna(), h["bid_price_3"].notna()],
    ["ask3", "bid3"],
    default="none",
)
ht = trades[trades["symbol"] == "HYDROGEL_PACK"].copy().sort_values(["day", "timestamp"])
ht = ht.merge(h[["day", "timestamp", "l3flag"]], on=["day", "timestamp"], how="left")
end_mid = h.groupby("day").tail(1).set_index("day")["mid_price"].to_dict()

def replay_hydrogel_policy(name, buy_cond, sell_cond, limit=200):
    rows = []
    for day, g in ht.groupby("day"):
        pos = 0
        cash = 0.0
        fills = 0
        qty = 0
        for _, r in g.iterrows():
            # buy from an aggressive seller by improving the visible bid
            if r["side"] == "sell_agg" and buy_cond(r) and pos < limit:
                q = int(min(r["quantity"], limit - pos))
                cash -= (r["bid_price_1"] + 1) * q
                pos += q
                fills += 1
                qty += q

            # sell to an aggressive buyer by improving the visible ask
            if r["side"] == "buy_agg" and sell_cond(r) and pos > -limit:
                q = int(min(r["quantity"], limit + pos))
                cash += (r["ask_price_1"] - 1) * q
                pos -= q
                fills += 1
                qty += q

        rows.append({
            "policy": name,
            "day": day,
            "pnl": cash + pos * end_mid[day],
            "fills": fills,
            "qty": qty,
            "end_pos": pos,
        })
    return rows

rows = []
rows += replay_hydrogel_policy("quote_both_all", lambda r: True, lambda r: True)
for span in [500, 1000, 2000, 5000]:
    rows += replay_hydrogel_policy(
        f"align_ema{span}_sign",
        lambda r, span=span: r[f"dev{span}"] < 0,
        lambda r, span=span: r[f"dev{span}"] > 0,
    )
    rows += replay_hydrogel_policy(
        f"align_ema{span}_sign_l3safe",
        lambda r, span=span: (r[f"dev{span}"] < 0) and (r["l3flag"] != "bid3"),
        lambda r, span=span: (r[f"dev{span}"] > 0) and (r["l3flag"] != "ask3"),
    )
rows += replay_hydrogel_policy(
    "wrong_ema2000_sign",
    lambda r: r["dev2000"] > 0,
    lambda r: r["dev2000"] < 0,
)

hgp_replay = pd.DataFrame(rows)
summary = hgp_replay.groupby("policy").agg(
    total=("pnl", "sum"),
    min_day=("pnl", "min"),
    fills=("fills", "sum"),
    qty=("qty", "sum"),
    end_pos_abs=("end_pos", lambda s: abs(s).sum()),
).sort_values("total", ascending=False)
display(summary.round(1))
display(hgp_replay.pivot(index="policy", columns="day", values="pnl").loc[
    ["quote_both_all", "align_ema5000_sign", "align_ema2000_sign", "align_ema1000_sign", "wrong_ema2000_sign"]
].round(1))

,total,min_day,fills,qty,end_pos_abs
policy,,,,,
align_ema5000_sign,52321.0,13133.0,469,1916,284
align_ema5000_sign_l3safe,51894.0,12718.0,462,1885,289
align_ema2000_sign,49219.0,10876.0,480,1974,150
align_ema2000_sign_l3safe,48813.0,10446.0,474,1948,160
align_ema1000_sign,44625.0,8505.0,475,1946,152
align_ema1000_sign_l3safe,44144.0,8075.0,468,1915,157
align_ema500_sign,41884.0,9607.0,483,1994,186
align_ema500_sign_l3safe,41403.0,9177.0,476,1963,189
quote_both_all,23282.0,-116.0,1010,4078,158


day,0,1,2
policy,,,
quote_both_all,10126.0,13272.0,-116.0
align_ema5000_sign,17860.0,21328.0,13133.0
align_ema2000_sign,16985.0,21358.0,10876.0
align_ema1000_sign,15865.0,20255.0,8505.0
wrong_ema2000_sign,-6859.0,-8086.0,-10992.0


### HYDROGEL interpretation

        The aggressive HYDROGEL takers are **liquidity-consuming mean-reversion losers**:

        - If the market is high vs a very slow EMA, sell to the buyer lifting the ask.
        - If the market is low vs a very slow EMA, buy from the seller hitting the bid.
        - Doing the opposite is deeply negative.

        This is more specific than "HYDROGEL has positive passive markout." It says which side to quote.
        The strongest replay variant is `align_ema5000_sign`: roughly 52k over three historical days, positive on
        every day, versus roughly 23k for quoting both sides.

## 2. HYDROGEL: book-state fingerprints

        L3 levels and quote-size imbalances are rare, so they should not be the main signal. They are still useful
        as bot-state diagnostics and quote filters.

In [4]:
display(Markdown("### L3 directional flag"))
l3 = h.groupby("l3flag").agg(
    n=("timestamp", "size"),
    ret10=("ret_10", "mean"),
    ret50=("ret_50", "mean"),
    ret100=("ret_100", "mean"),
    ret500=("ret_500", "mean"),
    ret1000=("ret_1000", "mean"),
    ret2000=("ret_2000", "mean"),
).round(3)
display(l3)
display(pd.crosstab(ht["l3flag"], ht["side"]))

display(Markdown("### Quote-size imbalance"))
h["l1_size_equal"] = h["bid_volume_1"] == h["ask_volume_1"]
h["l2_size_equal"] = h["bid_volume_2"] == h["ask_volume_2"]
h["l1_imb"] = h["ask_volume_1"].abs() - h["bid_volume_1"].abs()
h["l2_imb"] = h["ask_volume_2"].abs() - h["bid_volume_2"].abs()
h["imb_bin"] = pd.cut(h["l1_imb"], [-100, -5, -1, 0, 1, 5, 100])
display(h.groupby("l1_size_equal").agg(
    n=("timestamp", "size"),
    ret100=("ret_100", "mean"),
    ret500=("ret_500", "mean"),
    ret1000=("ret_1000", "mean"),
).round(3))
display(h.groupby("imb_bin", observed=False).agg(
    n=("timestamp", "size"),
    ret100=("ret_100", "mean"),
    ret1000=("ret_1000", "mean"),
).round(3))

### L3 directional flag

,n,ret10,ret50,ret100,ret500,ret1000,ret2000
l3flag,,,,,,,
ask3,516,3.856,4.262,4.660,5.530,7.143,8.897
bid3,474,-3.622,-2.923,-2.659,-1.463,5.846,8.186
none,29010,-0.004,0.013,0.123,1.278,3.412,7.783


side,buy_agg,sell_agg
l3flag,,
ask3,11,2
bid3,10,8
none,503,476


### Quote-size imbalance

,n,ret100,ret500,ret1000
l1_size_equal,,,,
False,961,1.185,2.177,6.362
True,29039,0.123,1.280,3.420


,n,ret100,ret1000
imb_bin,,,
"(-100, -5]",319,4.044,5.720
"(-5, -1]",182,5.506,9.699
"(-1, 0]",29039,0.123,3.420
"(0, 1]",33,-3.470,-5.571
"(1, 5]",201,-1.255,8.076
"(5, 100]",226,-3.500,4.769


### HYDROGEL book-state interpretation

        - `ask3` is a short-horizon up flag. If it appears, selling into an aggressive buyer is more dangerous.
        - `bid3` is a short-horizon down flag. If it appears, buying from an aggressive seller is more dangerous.
        - L1/L2 quote-size imbalances are rare. When they occur, they mostly confirm temporary one-sided pressure.

        These are quote filters, not standalone taker signals. The slow-EMA side selection is still the main edge.

## 3. OTM vouchers: coordinated strip seller

        This section tests whether the OTM voucher flow is independent per strike. It is not.
        The same timestamp and quantity repeatedly appears across multiple OTM strikes, which implies a single
        coordinated seller process or a tightly synchronized bot family.

In [5]:
otm_syms = ["VEV_5200", "VEV_5300", "VEV_5400", "VEV_5500", "VEV_6000", "VEV_6500"]
ot = trades[trades["symbol"].isin(otm_syms) & (trades["side"] == "sell_agg")].copy()
parent = ot.groupby(["day", "timestamp"]).agg(
    n=("symbol", "size"),
    qty_nunique=("quantity", "nunique"),
    qty=("quantity", "first"),
    symbols=("symbol", lambda s: tuple(sorted(s))),
).reset_index()
parent["dt"] = parent.groupby("day")["timestamp"].diff()

display(Markdown("### Parent event fingerprints"))
display(pd.DataFrame({
    "metric": ["parent_events", "same_quantity_rate", "median_interarrival", "p90_interarrival"],
    "value": [
        len(parent),
        (parent["qty_nunique"] == 1).mean(),
        parent["dt"].median(),
        parent["dt"].quantile(0.90),
    ],
}))
display(parent["symbols"].value_counts().head(12).to_frame("events"))
display(parent["qty"].value_counts().sort_index().to_frame("events"))

display(Markdown("### Inclusion rates"))
inc = []
for sym in otm_syms:
    inc.append({
        "symbol": sym,
        "events_present": int(parent["symbols"].apply(lambda tup, sym=sym: sym in tup).sum()),
        "rate": parent["symbols"].apply(lambda tup, sym=sym: sym in tup).mean(),
    })
display(pd.DataFrame(inc).round(3))

### Parent event fingerprints

,metric,value
0,parent_events,284.0
1,same_quantity_rate,1.0
2,median_interarrival,7900.0
3,p90_interarrival,22200.0


,events
symbols,
"(VEV_5400, VEV_5500, VEV_6000, VEV_6500)",119
"(VEV_5300, VEV_5400, VEV_5500, VEV_6000, VEV_6500)",81
"(VEV_5500, VEV_6000, VEV_6500)",27
"(VEV_5300, VEV_5500, VEV_6000, VEV_6500)",23
"(VEV_5200, VEV_5300, VEV_5400, VEV_5500, VEV_6000, VEV_6500)",9
"(VEV_5400, VEV_6000, VEV_6500)",8
"(VEV_5200, VEV_5400, VEV_5500, VEV_6000, VEV_6500)",5
"(VEV_6000, VEV_6500)",4
"(VEV_5300, VEV_5400, VEV_6000, VEV_6500)",3


,events
qty,
2,76
3,56
4,78
5,74


### Inclusion rates

,symbol,events_present,rate
0,VEV_5200,17,0.060
1,VEV_5300,119,0.419
2,VEV_5400,225,0.792
3,VEV_5500,267,0.940
4,VEV_6000,284,1.000
5,VEV_6500,284,1.000


### OTM seller role

        This is not six independent sellers. The bot is a **strip seller**:

        - `VEV_6000` and `VEV_6500` are included in every parent event.
        - `VEV_5500` is included in almost every parent event.
        - `VEV_5400` is included in most parent events.
        - `VEV_5300` and especially `VEV_5200` are conditional add-ons.
        - Quantity is identical across every strike inside a parent event.

        That is a strong bot fingerprint, but it is not automatically a huge edge because improving the bid by one
        tick consumes most of the OTM markout.

In [6]:
display(Markdown("### OTM seller event markouts: buy at bid vs bid+1"))
rows = []
for improve in [0, 1]:
    for sym in ["VEV_5200", "VEV_5300", "VEV_5400", "VEV_5500"]:
        ev = ot[ot["symbol"] == sym].copy()
        px = ev["bid_price_1"] + improve
        q = ev["quantity"]
        rows.append({
            "sym": sym,
            "price": f"bid+{improve}",
            "n": len(ev),
            "qty": int(q.sum()),
            "mk100": ((ev["fut_mid_100"] - px) * q).sum() / q.sum(),
            "mk500": ((ev["fut_mid_500"] - px) * q).sum() / q.sum(),
            "mk1000": ((ev["fut_mid_1000"] - px) * q).sum() / q.sum(),
            "mk2000": ((ev["fut_mid_2000"] - px) * q).sum() / q.sum(),
            "mk5000": ((ev["fut_mid_5000"] - px) * q).sum() / q.sum(),
        })
display(pd.DataFrame(rows).round(3))

display(Markdown("### Position-limited passive capture with terminal liquidation"))
end_mid_by_symbol = prices.groupby(["product", "day"]).tail(1).set_index(["product", "day"])["mid_price"].to_dict()
res = []
for sym in ["VEV_5200", "VEV_5300", "VEV_5400", "VEV_5500"]:
    for improve in [0, 1]:
        total = 0.0
        fills = 0
        qty = 0
        days = []
        ev = ot[ot["symbol"] == sym].sort_values(["day", "timestamp"])
        for day, g in ev.groupby("day"):
            pos = 0
            cash = 0.0
            f = 0
            qsum = 0
            for _, r in g.iterrows():
                if pos < 300:
                    q = int(min(r["quantity"], 300 - pos))
                    cash -= (r["bid_price_1"] + improve) * q
                    pos += q
                    f += 1
                    qsum += q
            pnl = cash + pos * end_mid_by_symbol[(sym, day)]
            total += pnl
            fills += f
            qty += qsum
            days.append(pnl)
        res.append({
            "sym": sym,
            "price": f"bid+{improve}",
            "total": total,
            "fills": fills,
            "qty": qty,
            "d0": days[0] if len(days) > 0 else 0,
            "d1": days[1] if len(days) > 1 else 0,
            "d2": days[2] if len(days) > 2 else 0,
        })
display(pd.DataFrame(res).round(1))

### OTM seller event markouts: buy at bid vs bid+1

,sym,price,n,qty,mk100,mk500,mk1000,mk2000,mk5000
0,VEV_5200,bid+0,17,62,1.218,4.065,9.056,7.548,8.065
1,VEV_5300,bid+0,119,414,0.797,0.761,2.214,2.431,-0.514
2,VEV_5400,bid+0,225,787,0.489,0.593,0.712,0.377,-0.799
3,VEV_5500,bid+0,267,937,0.499,0.509,0.565,0.430,-0.223
4,VEV_5200,bid+1,17,62,0.218,3.145,8.218,6.758,7.661
5,VEV_5300,bid+1,119,414,-0.184,-0.179,1.315,1.653,-1.005
6,VEV_5400,bid+1,225,787,-0.495,-0.372,-0.198,-0.454,-1.320
7,VEV_5500,bid+1,267,937,-0.493,-0.463,-0.342,-0.384,-0.732


### Position-limited passive capture with terminal liquidation

,sym,price,total,fills,qty,d0,d1,d2
0,VEV_5200,bid+0,1376.0,17,62,97.5,334.5,944.0
1,VEV_5200,bid+1,1314.0,17,62,82.5,313.5,918.0
2,VEV_5300,bid+0,3410.0,119,414,114.0,1002.0,2294.0
3,VEV_5300,bid+1,2996.0,119,414,-14.0,873.0,2137.0
4,VEV_5400,bid+0,2420.0,225,787,-212.0,612.0,2020.0
5,VEV_5400,bid+1,1633.0,225,787,-430.0,326.0,1737.0
6,VEV_5500,bid+0,865.5,255,881,65.5,135.0,665.0
7,VEV_5500,bid+1,-15.5,255,881,-215.5,-165.0,365.0


### OTM execution conclusion

        The strip seller is real and specific, but the edge is modest unless we can sit at the existing bid:

        - `VEV_5300` is the best OTM strip candidate.
        - `VEV_5400` has positive bid capture, but one-tick improvement reduces it sharply.
        - `VEV_5500` is almost erased by `bid+1`.
        - `VEV_6000/6500` remain confirmed zero-EV traps.

        This is not the 100k edge by itself. It is useful as a supporting long-biased sleeve, not the main fix.

## 4. Cross-strike and timestamp-pattern traps

        The data contains visually strong recurring windows and cross-strike residuals. We should not promote them
        blindly. The tests below are included to document what was checked and why most of it is secondary.

In [7]:
display(Markdown("### Cross-strike residual day-2 holdout search"))
active = ["VELVETFRUIT_EXTRACT", "VEV_4000", "VEV_4500", "VEV_5000", "VEV_5100", "VEV_5200", "VEV_5300", "VEV_5400", "VEV_5500"]
wide = prices[prices["product"].isin(active)].pivot_table(index=["day", "timestamp"], columns="product", values="mid_price")
med_spread = prices[prices["product"].isin(active)].groupby("product")["spread"].median().to_dict()
rows = []
for i, a in enumerate(active):
    for b in active[i + 1:]:
        df = wide[[a, b]].dropna().reset_index()
        train = df[df["day"].isin([0, 1])]
        test = df[df["day"] == 2]
        if len(train) < 1000 or len(test) < 1000:
            continue
        x = train[b].values
        y = train[a].values
        beta = np.cov(x, y)[0, 1] / np.var(x)
        alpha = y.mean() - beta * x.mean()
        resid = test[a] - (alpha + beta * test[b])
        change = resid.shift(-500) - resid
        sd = resid.std()
        mu = resid.mean()
        hi = resid > mu + 1.5 * sd
        lo = resid < mu - 1.5 * sd
        score = np.nan
        if hi.sum() > 20 and lo.sum() > 20:
            score = (-change[hi].mean() + change[lo].mean()) / 2
        cost = (med_spread[a] + abs(beta) * med_spread[b]) / 2
        rows.append({
            "pair": f"{a} ~ {beta:.2f}*{b}",
            "test_score500": score,
            "cost_proxy": cost,
            "hi_n": int(hi.sum()),
            "lo_n": int(lo.sum()),
        })
resid_search = pd.DataFrame(rows).sort_values("test_score500", ascending=False)
display(resid_search.head(15).round(3))

display(Markdown("### Timestamp phase day-holdout"))
phase_rows = []
for sym in ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT", "VEV_4000", "VEV_4500", "VEV_5000", "VEV_5300", "VEV_5400", "VEV_5500"]:
    g = prices[prices["product"] == sym].dropna(subset=["ret_1000"]).copy()
    g["phase10k"] = (g["timestamp"] // 10000).astype(int)
    train = g[g["day"].isin([0, 1])]
    test = g[g["day"] == 2]
    phase = train.groupby("phase10k")["ret_1000"].mean()
    pred = test["phase10k"].map(phase).fillna(0)
    y = test["ret_1000"]
    corr = np.corrcoef(pred, y)[0, 1] if pred.std() > 0 and y.std() > 0 else np.nan
    sign_acc = ((pred > 0) == (y > 0)).mean()
    active_mask = pred.abs() > phase.abs().median()
    edge = (np.sign(pred[active_mask]) * y[active_mask]).mean() if active_mask.any() else np.nan
    phase_rows.append({
        "symbol": sym,
        "corr_d2": corr,
        "sign_acc_d2": sign_acc,
        "edge_when_active": edge,
        "active_n": int(active_mask.sum()),
    })
display(pd.DataFrame(phase_rows).sort_values("edge_when_active", ascending=False).round(3))

### Cross-strike residual day-2 holdout search

,pair,test_score500,cost_proxy,hi_n,lo_n
14,VEV_4000 ~ 6.98*VEV_5500,8.980,13.988,1022,649
20,VEV_4500 ~ 6.98*VEV_5500,8.924,11.488,1039,640
7,VELVETFRUIT_EXTRACT ~ 6.97*VEV_5500,8.893,5.986,1019,644
19,VEV_4500 ~ 3.24*VEV_5400,8.708,9.618,725,718
13,VEV_4000 ~ 3.24*VEV_5400,8.663,12.118,735,716
6,VELVETFRUIT_EXTRACT ~ 3.23*VEV_5400,8.468,4.117,739,724
25,VEV_5000 ~ 6.82*VEV_5500,8.291,6.412,1023,636
24,VEV_5000 ~ 3.18*VEV_5400,7.776,4.589,720,733
29,VEV_5100 ~ 6.64*VEV_5500,6.953,5.320,1021,616
28,VEV_5100 ~ 3.08*VEV_5400,6.517,3.542,672,699


### Timestamp phase day-holdout

,symbol,corr_d2,sign_acc_d2,edge_when_active,active_n
0,HYDROGEL_PACK,0.043,0.547,5.095,4500
7,VEV_5500,-0.088,0.519,-0.121,4500
6,VEV_5400,-0.122,0.491,-0.333,4500
5,VEV_5300,-0.157,0.485,-1.344,4500
4,VEV_5000,-0.175,0.481,-3.428,4500
1,VELVETFRUIT_EXTRACT,-0.173,0.477,-3.612,4500
2,VEV_4000,-0.172,0.480,-3.621,4500
3,VEV_4500,-0.173,0.480,-3.634,4500


## Final read from notebook 05

        Strong enough to drive the next strategy design:

        - HYDROGEL has a website-style passive edge if we quote only the side aligned with slow mean reversion.
        - The four HYDROGEL roles are now specific: symmetric L1 maker, symmetric L2 wall maker, high-price buyer/liquidity taker, low-price seller/liquidity taker.
        - OTM vouchers have a coordinated strip seller, but the monetizable edge is concentrated in `VEV_5300` and partially `VEV_5400`.

        Not strong enough to be the main strategy:

        - Quote-size imbalance by itself.
        - L3 by itself.
        - Timestamp phase by itself.
        - Cross-strike residuals after transaction-cost proxy, except as a later pair-arb research path.